In [1]:
import pandas as pd
import os
import pickle

# Path to WSIs (root dir)
ROOT_DIR = "//regsj.intern/appl/Deep_Visual_Proteomics"

# Path to cache file
CACHE_FILE = "wsi_cache_christine.pkl"


class WSIStats:
    def __init__(self, filename):
        self.filename = filename
        self.rekvnr = os.path.basename(filename)[:8]
        self.file_size = os.path.getsize(filename)
        self.data_folder = os.path.splitext(filename)[0]
        self.data_folder_size = self.get_folder_size(self.data_folder) if os.path.isdir(self.data_folder) else 0

    @staticmethod
    def get_folder_size(folder):
        total = 0
        for dirpath, dirnames, filenames in os.walk(folder):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                if os.path.isfile(fp):
                    total += os.path.getsize(fp)
        return total

    def to_dict(self):
        return {
            "filename": self.filename,
            "rekvnr": self.rekvnr,
            "file_size": self.file_size,
            "data_folder_size": self.data_folder_size,
        }

class WSIStatsCache:
    def __init__(self, root_dir, cache_file):
        self.root_dir = root_dir
        self.cache_file = cache_file
        self.stats = []

    def scan_files(self):
        cached_files = set(stat.filename for stat in self.stats)
        new_stats = []
        for dirpath, dirnames, filenames in os.walk(self.root_dir):
            for fname in filenames:
                if fname.lower().endswith('.mrxs'):
                    fpath = os.path.join(dirpath, fname)
                    if fpath not in cached_files:
                        new_stats.append(WSIStats(fpath))
        if new_stats:
            print(f"Found {len(new_stats)} new files.")
        self.stats.extend(new_stats)

    def save_cache(self):
        with open(self.cache_file, "wb") as f:
            pickle.dump(self.stats, f)

    def load_cache(self):
        if os.path.exists(self.cache_file):
            with open(self.cache_file, "rb") as f:
                self.stats = pickle.load(f)
            return True
        return False

    def get_stats_dicts(self):
        return [stat.to_dict() for stat in self.stats]

def main(reload=True):
    cache = WSIStatsCache(ROOT_DIR, CACHE_FILE)
    cache.load_cache()
    if reload:
        cache.scan_files()
        cache.save_cache()
    stats_dicts = cache.get_stats_dicts()
    # Create DataFrame with filenames as index
    df = pd.DataFrame(stats_dicts)
    df.set_index("filename", inplace=True)
    return df

In [2]:
# Path to pathology metadata
df_path = "D:\DATA\initial_cleaning.xlsx"
df_pathology = pd.read_excel(df_path)

# Path to WSI stats cache
df_wsi = main(reload = False)

AttributeError: 'WSIStats' object has no attribute 'date_modified'

In [ ]:
# Check for WSI with no associated data
missing_data = df_wsi[
    (df_wsi["file_size"].isna() | (df_wsi["file_size"] == 0)) |
    (df_wsi["data_folder_size"].isna() | (df_wsi["data_folder_size"] == 0))
]

print("WSI with missing data: ", missing_data)

In [ ]:
# Drop rows with no associated data
rows_to_drop = missing_data[missing_data.any(axis=1)].index
df_no_missing = df_wsi.drop(index=rows_to_drop)

print(f"Original rows: {len(df_wsi)}, After dropping: {len(df_no_missing)}")

In [ ]:
print(df_pathology["rekvnr"].head())
print(df_no_missing["rekvnr"].head())

In [ ]:
df_no_missing["rekvnr"] = pd.to_numeric(df_no_missing["rekvnr"], errors="coerce").astype("Int64")

In [ ]:
rekvnr_pathology = set(df_pathology["rekvnr"])
rekvnr_wsi = set(df_no_missing["rekvnr"])

overlap = rekvnr_pathology & rekvnr_wsi
only_in_pathology = rekvnr_pathology - rekvnr_wsi
only_in_wsi = rekvnr_wsi - rekvnr_pathology

print(f"rekvnr in both: {len(overlap)}")
print(f"rekvnr only in pathology: {len(only_in_pathology)}")
print(f"rekvnr only in WSI: {len(only_in_wsi)}")

In [ ]:
count_in_wsi = df_no_missing["rekvnr"].isin(overlap).sum()
print(f"Number of rows in WSI with overlapping rekvnr: {count_in_wsi}")

counts_per_rekvnr = df_no_missing[df_no_missing["rekvnr"].isin(overlap)]["rekvnr"].value_counts()
print(counts_per_rekvnr)

In [ ]:
# Subset pathology DataFrame to only overlapping rekvnr
df_overlap = df_pathology[df_pathology["rekvnr"].isin(overlap)].copy()

# Count how many WSI files exist for each rekvnr
wsi_counts = df_no_missing["rekvnr"].value_counts()

# Add the count as a new column in the pathology subset
df_overlap["wsi count"] = df_overlap["rekvnr"].map(wsi_counts).fillna(0).astype(int)

print(df_overlap.head())

In [ ]:
# Create a mapping from rekvnr to list of filenames
filenames_per_rekvnr = df_no_missing.groupby("rekvnr").apply(
    lambda g: list(g.index),
    include_groups=False
)

# Add the list of filenames as a new column in df_overlap
df_overlap["wsi filenames"] = df_overlap["rekvnr"].map(filenames_per_rekvnr)

print(df_overlap[["rekvnr", "wsi count", "wsi filenames"]].head())

In [ ]:
# Save to Excel
output_file = "D:\DATA\overlapping_rekvnr.xlsx"
df_overlap.to_excel(output_file, index=False)

print(f"Saved DataFrame to {output_file}")